In [11]:
import pandas as pd

# Path to the CSV file
csv_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\merged_encoded_latin1_processed.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path, encoding='latin-1')


# Display the DataFrame
print("Contents of the CSV file:")
print(df)

Contents of the CSV file:
   CAS-Number             Container Name  \
0    108-88-3                     Toluol   
1     75-05-8                Acetonitril   
2     50-00-0  Formaldehyd-Lösung >= 25%   
3  14808-60-7                      Quarz   
4  14808-60-7                      Quarz   
5  14808-60-7                      Quarz   

                                    Title  Location   Inventory-ID  \
0                                  Toluol  3201/275  21-523-0138-7   
1                             Acetonitril  3201/275  21-523-0014-4   
2                     Formaldehyd -Lösung  3201/275  22-523-0001-3   
3                                   Quarz  3201/275  20-523-0082-4   
4                         Quarzwolle SiO2  3201/275  20-523-0094-7   
5  Quarz feinkörnig gewaschen und geglüht  3201/275  20-523-0107-4   

  Structure formula          Manufacturer                        Purity  
0             C7 H8  Sigma-Aldrich 244511              anhydrous, 99.8%  
1          C2 H3 N1      A

In [12]:
import os
import time
import datetime
import csv
import json
import elabapi_python
from elabapi_python.rest import ApiException
import pandas as pd

#########################
#         CONFIG        #
#########################
API_HOST_URL = 'https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2'
# replace with your api key
API_KEY = '23-b3f02a4d01ec10b4dcba33596866a2435a7baaffc6252f7e61b57ab88a20db5a28364e8a77dc50b2291523'
#########################
#      END CONFIG       #
#########################

# Configure the api client
configuration = elabapi_python.Configuration()
configuration.api_key['api_key'] = API_KEY
configuration.api_key_prefix['api_key'] = 'Authorization'
configuration.host = API_HOST_URL
configuration.debug = False
configuration.verify_ssl = False

# Create an instance of the API class
api_client = elabapi_python.ApiClient(configuration)
# Fix issue with Authorization header not being properly set by the generated lib
api_client.set_default_header(header_name='Authorization', header_value=API_KEY)

# Load the items API
items_api = elabapi_python.ItemsApi(api_client)

# Path to the CSV file
csv_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\merged_encoded_latin1_processed.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path, encoding='latin-1')

# Category ID in eLab where the items will be created
category_id = 238  # Adjust this according to your category ID in eLab

# Base JSON for metadata
base_json = '''{
    "elabftw": {
        "display_main_text": false
    },
    "extra_fields": {
        "Inventory-ID": {
            "type": "text",
            "value": ""
        },
        "Container Number": {
            "type": "text",
            "value": ""
        },
        "Location": {
            "type": "text",
            "value": ""
        },
        "Purity": {
            "type": "text",
            "value": ""
        },
        "Manufacturer": {
            "type": "text",
            "value": ""
        },
        "CAS-Number": {
            "type": "text",
            "value": ""
        },
        "Batch Number": {
            "type": "text",
            "value": ""
        },
        "Opening Date": {
            "type": "text",
            "value": ""
        },
        "Structure formula": {
            "type": "text",
            "value": ""
        }
    }
}'''

metadata_as_json = json.loads(base_json)

# Iterate over the first 6 rows in the DataFrame
for index, row in df.iterrows():
    try:
        # Create an item in the correct category
        response = items_api.post_item_with_http_info(body={'category_id': category_id, 'title': row['Title']})
        # Extract the newly created item ID
        location_header_in_response = response[2].get('Location')
        print(f'The newly created item is here: {location_header_in_response}')
        item_id = int(location_header_in_response.split('/').pop())

        # Update metadata with values from the current row
        metadata_as_json = json.loads(base_json)  # Load base JSON for metadata
        metadata_as_json['extra_fields']['Inventory-ID']['value'] = row['Inventory-ID']
        metadata_as_json['extra_fields']['Container Number']['value'] = row['Container Name']
        metadata_as_json['extra_fields']['Location']['value'] = row['Location']
        metadata_as_json['extra_fields']['Purity']['value'] = row['Purity']
        metadata_as_json['extra_fields']['CAS-Number']['value'] = row['CAS-Number']

        # Check if Manufacturer value is empty
        if pd.isnull(row['Manufacturer']) or row['Manufacturer'].strip() == '':
            metadata_as_json['extra_fields']['Manufacturer']['value'] = 'Unknown'
        else:
            metadata_as_json['extra_fields']['Manufacturer']['value'] = row['Manufacturer']

        # Check if Purity value is missing
        if pd.isnull(row['Purity']) or row['Purity'].strip() == '':
            metadata_as_json['extra_fields']['Purity']['value'] = 'Not specified'

        # Check if Structure formula value is missing
        if pd.isnull(row['Structure formula']) or row['Structure formula'].strip() == '':
            metadata_as_json['extra_fields']['Structure formula']['value'] = 'Not specified'
        else:
            metadata_as_json['extra_fields']['Structure formula']['value'] = row['Structure formula']

        # Modify the item with the correct metadata
        items_api.patch_item(item_id, body={'title': row['Title'], 'metadata': json.dumps(metadata_as_json)})

        print(f"Item '{row['Title']}' created successfully.")
    except ApiException as e:
        print(f"An error occurred while creating item '{row['Title']}': {e}")


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/7174
Item 'Toluol' created successfully.
The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/7175


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Item 'Acetonitril' created successfully.


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


...

Item 'Quarz' created successfully.
The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/7178


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Item 'Quarzwolle SiO2' created successfully.
The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/7179


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Item 'Quarz feinkörnig gewaschen und geglüht' created successfully.
